In [0]:
dbutils.help()
dbutils.secrets()
dbutils.secrets.listScopes()
dbutils.secrets.list(scope="text scope")

In [0]:
spark.conf.set(
    "fs.azure.account.auth.type.insurancedatastorage.dfs.core.windows.net",
    "OAuth"
)

In [0]:
spark.conf.set(
    "fs.azure.account.oauth2.client.id.insurancedatastorage.dfs.core.windows.net",
    dbutils.secrets.get(scope="text scope", key="clientid")
)

In [0]:
spark.conf.set(
    "fs.azure.account.oauth2.client.secret.insurancedatastorage.dfs.core.windows.net",
    dbutils.secrets.get(scope="text scope", key="password"))

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.insurancedatastorage.dfs.core.windows.net",
    "https://login.microsoftonline.com/bda11e34-4864-4eb6-82cb-acbfb3913d55/oauth2/v2.0/token"
)


In [0]:
#data loading to the notebook variable from bronze container
agent_df = spark.read.csv(
    "abfss://bronze@insurancedatastorage.dfs.core.windows.net/agentinfo.csv",
    header=True,
    inferSchema=True
)
customermaster_df = spark.read.csv(
    "abfss://bronze@insurancedatastorage.dfs.core.windows.net/customermaster.csv",
    header=True,
    inferSchema=True
)
claimhistory_df = spark.read.csv(
    "abfss://bronze@insurancedatastorage.dfs.core.windows.net/claimhistory.csv",
    header=True,
    inferSchema=True
)
policydetails_df = spark.read.csv(
    "abfss://bronze@insurancedatastorage.dfs.core.windows.net/policydetails.csv",
    header=True,
    inferSchema=True
)
customerfeedback_df = spark.read.csv(
    "abfss://bronze@insurancedatastorage.dfs.core.windows.net/customerfeedback.csv",
    header=True,
    inferSchema=True
)
#display(agent_df)


In [0]:
# drop duplicated from each table
agent_df = agent_df.dropDuplicates()
customermaster_df = customermaster_df.dropDuplicates()
claimhistory_df = claimhistory_df.dropDuplicates()
policydetails_df = policydetails_df.dropDuplicates()
customerfeedback_df = customerfeedback_df.dropDuplicates()


In [0]:
#Check distinct value of region column agent table
agent_df.select("region").distinct().show()

#trasformation from date time to date using cast(DateType())
from pyspark.sql.types import DateType
agent_df = agent_df.withColumn("join_date", agent_df["join_date"].cast(DateType()))
#display(agent_df)

In [0]:
#trasformation of region column in customer master
from pyspark.sql.functions import regexp_replace

customermaster_df = customermaster_df.withColumn(
    "region",
    regexp_replace("region", "souht", "South")
)
customermaster_df=customermaster_df.withColumn(
    "region",regexp_replace("region","west","West")
) 
#display(customermaster_df.select("region").distinct())

In [0]:
#data transformation of date_joined column from datetime to date
customermaster_df=customermaster_df.withColumn("date_joined", customermaster_df["date_joined"].cast(DateType()))
#display(customermaster_df)

In [0]:

#data transformation of gender column
from pyspark.sql.functions import col, when
customermaster_df = customermaster_df.withColumn(
    "gender",
    when(col("gender") == "Mal", "Male")
    .when(col("gender") == "females", "Female")
    .when(col("gender") == "males", "Male")
    .otherwise(col("gender"))
)

In [0]:
#distinct value of various column
display(customermaster_df.select("gender").distinct())
display(customermaster_df.select("marital_status").distinct())
display(customermaster_df.select("occupation").distinct())

In [0]:
display(customermaster_df)

In [0]:
from pyspark.sql.types import DateType
claimhistory_df = claimhistory_df.withColumn("claim_date", claimhistory_df["claim_date"].cast(DateType()))

In [0]:
#Check distinct value of claim_status and replace with correct value
display(claimhistory_df.limit(5))
display(claimhistory_df.select("claim_status").distinct())
from pyspark.sql.functions import col, when
claimhistory_df=claimhistory_df.withColumn(
    "claim_status",
    when(col("claim_status") == "Aproval", "Approved")
    .when(col("claim_status") == "Rejeted", "Rejected")
    .otherwise(col("claim_status"))
)

#Check distinct value of claim_type, claim_type
display(claimhistory_df.select("claim_type").distinct())
display(claimhistory_df.select("claim_type").distinct())

In [0]:
display(policydetails_df.limit(5))

In [0]:
from pyspark.sql.functions import months_between, round, col
from pyspark.sql.types import DateType

policydetails_df = spark.read.csv(
    "abfss://bronze@insurancedatastorage.dfs.core.windows.net/policydetails.csv",
    header=True,
    inferSchema=True
).dropDuplicates().withColumn(
    "policy_start_date", col("policy_start_date").cast(DateType())
).withColumn(
    "policy_end_date", col("policy_end_date").cast(DateType())
)

policydetails_df = policydetails_df.withColumn(
    "duration_in_year",
    round(months_between("policy_end_date", "policy_start_date") / 12, 1)
)

#display(policydetails_df.select("policy_start_date", "policy_end_date", "duration_in_year"))

In [0]:
#data transformation of product_type value
from pyspark.sql.functions import regexp_replace

policydetails_df = policydetails_df.withColumn(
    "product_type",
    regexp_replace("product_type", "Properties", "Property")
)

display(policydetails_df.limit(5))
display(policydetails_df.select("product_type").distinct())
display(policydetails_df.select("status").distinct())

In [0]:
#check the customer feedback data
display(customerfeedback_df.limit(5))

In [0]:
# Write transformed DataFrames to the silver container in Parquet format
agent_df.write.mode("overwrite").parquet("abfss://silver@insurancedatastorage.dfs.core.windows.net/agentinfo/")
customermaster_df.write.mode("overwrite").parquet("abfss://silver@insurancedatastorage.dfs.core.windows.net/customermaster/")
claimhistory_df.write.mode("overwrite").parquet("abfss://silver@insurancedatastorage.dfs.core.windows.net/claimhistory/")
policydetails_df.write.mode("overwrite").parquet("abfss://silver@insurancedatastorage.dfs.core.windows.net/policydetails/")
customerfeedback_df.write.mode("overwrite").parquet("abfss://silver@insurancedatastorage.dfs.core.windows.net/customerfeedback/")